In [ ]:
using Bessels, MPSDynamics
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/simulator.jl");
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/visualization/plot_settings.jl");
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/visualization/plot_utils.jl");
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/baths.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 3.7*2*pi
ω_b_0 = 7.05*2*pi

Lamb_shift = - 0.05 #Lamb shift in GHz


φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.2
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b_0
E_Cb = C_b_inv/2

E_J = 37*2*pi


##Drive


ω_d=ω_b_0#+Lamb_shift*2*pi # Drive frequency
ω_p=2ω_a-ω_d

ϵ_0 = 0.25
g2_max = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10*g2_max

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10; # 1/ns from Leon's paper
κ_2 = 4*g2_max^2/κ_b
κ_a = 3e-3*κ_2 #from Leon's paper

In [ ]:
n=7000

bath=Exp_decay
bath_params =(Ω=50*2*pi, ω_cut_off=30*2*pi)

AB, J = bath(bath_params=bath_params,κ=κ_b, C_inv=C_b_inv)
chain_params = chaincoeffs_finiteT(n, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=7000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0

In [ ]:
t_max=15/κ_b
ϕ_a_0, q_a_0=0.0, 0.0
ϕ_b_0, q_b_0=0.0, 0.0

system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
drive = DriveTwoModes(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = OneChain(n, chain_params...)

ϕ,ϕ_dot,t_list,Mq_chain,Mϕ_chain = simulator(t_max, system, drive, environment, solver=emission)

In [ ]:
A_chain = Mq_chain * Mϕ_chain

vals_chain, vecs_chain = eigen(A_chain)
ωk²_chain = real(vals_chain)
ωk_chain = sqrt.(ωk²_chain)
P_chain = real(vecs_chain)


Dq_chain = diag(P_chain'*(Mq_chain \ P_chain))  # uses linear solve instead of inv
Dϕ_chain = diag(P_chain' * Mϕ_chain * P_chain)

In [ ]:
# Solve P_chain * S = ϕ for S instead of inverting
S = P_chain \ ϕ
S_dot = P_chain \ ϕ_dot

# Energy in normal modes
Energies_normal_modes_chain = Dq_chain .* S_dot.^2 .+ Dϕ_chain .* S.^2;


In [ ]:
f_a=ω_a/2pi
f_b=ω_b_0/2pi
f_p=abs(ω_p/2pi)
plot(ωk_chain/2pi,Energies_normal_modes_chain)
vline!([f_a],linestyle=:dash, color=:red)
annotate!((f_a)-.2,1e-0, Plots.text(L"\omega_d/2+\omega_p/2",18, :red, rotation = 90 ))
vline!([f_a-2f_p],linestyle=:dash, color=:red)
annotate!((f_a-2f_p)-.2,1e-0, Plots.text(L"\omega_d/2-3\omega_p/2",18, :red, rotation = 90 ))
vline!([f_a+2f_p],linestyle=:dash, color=:red)
annotate!((f_a+2f_p)-.2,1e-0, Plots.text(L"\omega_d/2+5\omega_p/2",18, :red, rotation = 90 ))
vline!([f_a-4f_p],linestyle=:dash, color=:red)
annotate!((f_a-4f_p)-.2,1e-0, Plots.text(L"\omega_d/2-7\omega_p/2",18, :red, rotation = 90 ))
vline!([f_a+4f_p],linestyle=:dash, color=:red)
annotate!((f_a+4f_p)-.2,1e-0, Plots.text(L"\omega_d/2+9\omega_p/2",18, :red, rotation = 90 ))

vline!([f_p],linestyle=:dash, color=:red)
annotate!(f_p-.2,1e-0, Plots.text(L"\omega_p", 18, :red, rotation = 90 ))
vline!([3f_p],linestyle=:dash, color=:red)
annotate!(3f_p-.2,1e-0, Plots.text(L"3\omega_p", 18, :red, rotation = 90 ))
vline!([f_b],linestyle=:dash, color=:red)

annotate!(f_b-.2,1e-8, Plots.text(L"\omega_d",18, :red, rotation = 90 ))
vline!([f_b-2f_p],linestyle=:dash, color=:red)
annotate!((f_b-2f_p)-.2,1e-8, Plots.text(L"\omega_d-2\omega_p",18, :red, rotation = 90 ))
vline!([f_b+2f_p],linestyle=:dash, color=:red)
annotate!((f_b+2f_p)-.2,1e-8, Plots.text(L"\omega_d+2\omega_p",18, :red, rotation = 90 ))
#vline!([f_b-4f_p],linestyle=:dash, color=:red)
#annotate!((f_b-4f_p)-.2,1e-8, Plots.text(L"\omega_d-4\omega_p",18, :red, rotation = 90 ))
vline!([f_b+4f_p],linestyle=:dash, color=:red)
annotate!((f_b+4f_p)-.2,1e-8, Plots.text(L"\omega_d+4\omega_p",18, :red, rotation = 90 ))

vline!([3f_b/2-5f_p/2],linestyle=:dash, color=:red)
annotate!((3f_b/2-5f_p/2)-.2,1e-8, Plots.text(L"3\omega_d/2-5\omega_p/2",18, :red, rotation = 90 ))
vline!([3f_b/2-f_p/2],linestyle=:dash, color=:red) 
annotate!((3f_b/2-f_p/2)-.2,1e-8, Plots.text(L"3\omega_d/2-\omega_p/2",18, :red, rotation = 90 ))
vline!([3f_b/2+3f_p/2],linestyle=:dash, color=:red)
annotate!((3f_b/2+3f_p/2)-.2,1e-8, Plots.text(L"3\omega_d/2+3\omega_p/2",18, :red, rotation = 90 ))
vline!([3f_b/2+7f_p/2],linestyle=:dash, color=:red)
annotate!((3f_b/2+7f_p/2)-.2,1e-8, Plots.text(L"3\omega_d/2+7\omega_p/2",18, :red, rotation = 90 ))

#vline!([2f_b-3f_p],linestyle=:dash, color=:red)
#annotate!((2f_b-3f_p)-.2,1e-8, Plots.text(L"2\omega_d-3\omega_p",18, :red, rotation = 90 ))
vline!([2f_b-f_p],linestyle=:dash, color=:red)
annotate!((2f_b-f_p)-.2,1e-8, Plots.text(L"2\omega_d-\omega_p",18, :red, rotation = 90 ))
vline!([2f_b+f_p],linestyle=:dash, color=:red)
annotate!((2f_b+f_p)-.2,1e-8, Plots.text(L"2\omega_d+\omega_p",18, :red, rotation = 90 ))

plot!(ylabel="Energy stored in the chain", xlabel="Frequency (GHz)", title = L"κt = "*"$(κ_b*t_max), "*L"n_{osc}="*"$(n)", legend=false, yaxis=:log, xlims=(0.1, 16), ylims=(1e-10, 1e2))
#savefig("../../Figures/Emission_spectrum/Energies_normal_modes_chain_two_photon_09_07.pdf")